# 06 - The ladder table and the gates

Reads every record on disk, joins it to the manifest (left-joined on the manifest,
so a planned-but-missing cell renders as `--` rather than vanishing), and writes
`table_ladder.csv` / `.tex` / `.txt` under `<results>/tables/`.

**Pass the tier explicitly.** `ladder.report()` otherwise falls back to `BACP_TIER`,
which defaults to 0, and tier 0 uses `smoke.*` keys that match none of the real
records - so you get an empty table and no error to tell you why.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


In [ ]:
TIER = 1
out = nb.report(TIER)


## Reading it

- **Noise floor** = median within-rung sd. `*` marks a delta above it, `~` below.
  A `~` is not a result. Nothing below the floor may be bolded.
- **Endpoint** is mean top-1 over the final 5 epochs, not best-epoch: best-of-a-noisy-
  sequence is upward-biased and higher-variance. No test-set model selection, no
  early stopping.
- A **STOP** halts the ladder by design. The next stage would spend GPU-hours
  measuring something the controls have already ruled out.


In [ ]:
agg = out['agg']
agg[['rung', 'n_ok', 'n_failed', 'n_missing', 'mean', 'sd']]


## Health check before believing any of it

A collapsed arm reads as a valid number. Two collapsed arms read as a delta of exactly zero - which is how G1 can report a pass on two dead runs.


In [ ]:
p = nb.progress(TIER)
if len(p):
    bad = p[p['nan'] | (p['acc'] < 15)]
    print(f'{len(bad)} of {len(p)} runs look collapsed or NaN')
    display(bad)
else:
    print('no logs yet')


## Experiment invariants

These assert against the *records*, not the code. No unit test can substitute for them.


In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-m', 'results', '-s',
                    'project/tests/test_experiment_invariants.py'],
                   cwd=str(info['repo']), capture_output=True, text=True)
print(p.stdout[-4000:])
